# Middlebury 离线双目实验

在 Python 3.11 (SVP_env) 内核运行。参数来自 config/middlebury.json；输入已校正，展示视差与轴向深度，误差只在非遮挡真值与固定搜索区域交集内计算。

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config/middlebury.json").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from stereo_vision.config import load_config
from stereo_vision.datasets import load_middlebury_scene
from stereo_vision.outputs import save_outputs
from stereo_vision.pipeline import reconstruct_scene

print("Kernel runtime:", sys.version.split()[0], "OpenCV", cv2.__version__)

In [ ]:
config = load_config(root / "config/middlebury.json")
cv2.setNumThreads(config.threads)
scene = load_middlebury_scene(config.data_root, config.scene, config.split)
result = reconstruct_scene(scene, config.sgbm)
result.metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(scene.pair.left_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Left rectified image")
fig.colorbar(axes[1].imshow(result.disparity.disparity_px, cmap="turbo"), ax=axes[1], label="Disparity (px)")
fig.colorbar(axes[2].imshow(result.reconstruction.depth_m, cmap="viridis"), ax=axes[2], label="Axial depth (m)")
for axis in axes:
    axis.axis("off")
fig.tight_layout()

In [ ]:
output_dir = config.output_root / config.split / scene.name
report = save_outputs(output_dir, scene, result, config)
print(output_dir)
report["stage_ms"]